In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load cleaned dataset
file_path = "Cleaned_Data_Set.xlsx"
df = pd.read_excel(file_path)

# Basic dataset information
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


# Date conversion
date_columns = ["Event_Date", "Consent_Date", "Revocation_Date"]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")


# Standardize text columns
text_columns = [
    "Data_Source",
    "Purpose",
    "Consent_Status",
    "Deletion_Status",
    "Extract_Status",
    "Audit_Status",
    "Region"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()


# Standardize important status values
if "Consent_Status" in df.columns:
    df["Consent_Status"] = df["Consent_Status"].str.title()

if "Audit_Status" in df.columns:
    df["Audit_Status"] = df["Audit_Status"].str.title()

if "Deletion_Status" in df.columns:
    df["Deletion_Status"] = df["Deletion_Status"].str.title()

if "Extract_Status" in df.columns:
    df["Extract_Status"] = df["Extract_Status"].str.title()


# Basic numerical summary
numeric_columns = df.select_dtypes(include=np.number).columns

print("\nNumerical Summary:")
print(df[numeric_columns].describe())


# Consent analysis
consent_summary = (
    df.groupby("Consent_Status", dropna=False)
      .agg(
          Records=("Record_ID", "count"),
          Customers=("Customer_ID", "nunique"),
          Records_Processed=("Records_Processed", "sum")
      )
      .reset_index()
)

print("\nConsent Summary:")
print(consent_summary)


# Consent rate
total_records = len(df)

consented_records = (
    df["Consent_Status"].eq("Granted").sum()
)

consent_rate = (
    consented_records / total_records * 100
    if total_records > 0 else 0
)

print("\nConsent Rate:", round(consent_rate, 2), "%")


# Records without valid consent
no_consent_records = df[
    ~df["Consent_Status"].eq("Granted")
].copy()

print("\nRecords Without Granted Consent:",
      len(no_consent_records))


# Potential analytics exposure
analytics_exposure = df[
    (df["Purpose"].eq("Analytics")) &
    (~df["Consent_Status"].eq("Granted"))
].copy()

print("\nAnalytics Records Without Granted Consent:",
      len(analytics_exposure))


# Revoked consent analysis
revoked_records = df[
    df["Consent_Status"].eq("Revoked")
].copy()

print("\nRevoked Consent Records:",
      len(revoked_records))


# Retention analysis
if "Retention_Days" in df.columns:

    retention_summary = (
        df.groupby("Retention_Days", dropna=False)
          .agg(
              Records=("Record_ID", "count"),
              Customers=("Customer_ID", "nunique")
          )
          .reset_index()
          .sort_values("Retention_Days")
    )

    print("\nRetention Summary:")
    print(retention_summary)


# Calculate retention expiry date
df["Retention_Expiry_Date"] = (
    df["Event_Date"] +
    pd.to_timedelta(df["Retention_Days"], unit="D")
)


# Retention status
today = pd.Timestamp.today().normalize()

df["Retention_Status"] = np.where(
    df["Retention_Expiry_Date"] < today,
    "Expired",
    "Within Retention"
)

print("\nRetention Status:")
print(df["Retention_Status"].value_counts())


# Deletion analysis
deletion_summary = (
    df.groupby("Deletion_Status", dropna=False)
      .agg(
          Records=("Record_ID", "count"),
          Customers=("Customer_ID", "nunique")
      )
      .reset_index()
)

print("\nDeletion Summary:")
print(deletion_summary)


# Pending deletion records
pending_deletion = df[
    df["Deletion_Status"].eq("Pending Deletion")
].copy()

print("\nPending Deletion Records:",
      len(pending_deletion))


# Deleted records
deleted_records = df[
    df["Deletion_Status"].eq("Deleted")
].copy()

print("\nDeleted Records:",
      len(deleted_records))


# Extract compliance analysis
extract_summary = (
    df.groupby("Extract_Status", dropna=False)
      .agg(
          Records=("Record_ID", "count"),
          Customers=("Customer_ID", "nunique")
      )
      .reset_index()
)

print("\nExtract Summary:")
print(extract_summary)


# Records still present in extracts
extract_exposure = df[
    (df["Deletion_Status"].isin(["Deleted", "Pending Deletion"])) &
    (df["Extract_Status"].eq("Present"))
].copy()

print("\nDeletion/Extract Exposure:",
      len(extract_exposure))


# Audit status analysis
audit_summary = (
    df.groupby("Audit_Status", dropna=False)
      .agg(
          Records=("Record_ID", "count"),
          Customers=("Customer_ID", "nunique"),
          Records_Processed=("Records_Processed", "sum")
      )
      .reset_index()
)

print("\nAudit Summary:")
print(audit_summary)


# Compliance rate
compliant_records = (
    df["Audit_Status"].eq("Compliant").sum()
)

compliance_rate = (
    compliant_records / total_records * 100
    if total_records > 0 else 0
)

print("\nCompliance Rate:",
      round(compliance_rate, 2), "%")


# Compliance issues
compliance_issues = df[
    ~df["Audit_Status"].eq("Compliant")
].copy()

print("\nCompliance Issues:",
      len(compliance_issues))


# Region-wise compliance
region_summary = (
    df.groupby("Region", dropna=False)
      .agg(
          Records=("Record_ID", "count"),
          Customers=("Customer_ID", "nunique"),
          Compliant_Records=(
              "Audit_Status",
              lambda x: (x == "Compliant").sum()
          )
      )
      .reset_index()
)

region_summary["Compliance_Rate"] = (
    region_summary["Compliant_Records"] /
    region_summary["Records"] * 100
)

region_summary["Compliance_Rate"] = (
    region_summary["Compliance_Rate"].round(2)
)

print("\nRegion-wise Compliance:")
print(region_summary)


# Purpose-wise compliance
purpose_summary = (
    df.groupby("Purpose", dropna=False)
      .agg(
          Records=("Record_ID", "count"),
          Customers=("Customer_ID", "nunique"),
          Compliant_Records=(
              "Audit_Status",
              lambda x: (x == "Compliant").sum()
          )
      )
      .reset_index()
)

purpose_summary["Compliance_Rate"] = (
    purpose_summary["Compliant_Records"] /
    purpose_summary["Records"] * 100
).round(2)

print("\nPurpose-wise Compliance:")
print(purpose_summary)


# Data source analysis
source_summary = (
    df.groupby("Data_Source", dropna=False)
      .agg(
          Records=("Record_ID", "count"),
          Customers=("Customer_ID", "nunique"),
          Records_Processed=("Records_Processed", "sum")
      )
      .reset_index()
)

print("\nData Source Summary:")
print(source_summary)


# Overall KPI summary
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Unique Customers",
        "Granted Consent Records",
        "Consent Rate",
        "Records Without Granted Consent",
        "Analytics Records Without Consent",
        "Revoked Consent Records",
        "Expired Retention Records",
        "Pending Deletion Records",
        "Deleted Records",
        "Deletion/Extract Exposure",
        "Compliant Records",
        "Compliance Rate",
        "Compliance Issues"
    ],
    "Value": [
        len(df),
        df["Customer_ID"].nunique(),
        consented_records,
        round(consent_rate, 2),
        len(no_consent_records),
        len(analytics_exposure),
        len(revoked_records),
        (df["Retention_Status"] == "Expired").sum(),
        len(pending_deletion),
        len(deleted_records),
        len(extract_exposure),
        compliant_records,
        round(compliance_rate, 2),
        len(compliance_issues)
    ]
})

print("\nKPI Summary:")
print(kpi_summary)


# Save one consolidated analysis result
output_file = "Python Analysis Result.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="Cleaned Data",
        index=False
    )

    kpi_summary.to_excel(
        writer,
        sheet_name="KPI Summary",
        index=False
    )

    consent_summary.to_excel(
        writer,
        sheet_name="Consent Analysis",
        index=False
    )

    retention_summary.to_excel(
        writer,
        sheet_name="Retention Analysis",
        index=False
    )

    deletion_summary.to_excel(
        writer,
        sheet_name="Deletion Analysis",
        index=False
    )

    extract_summary.to_excel(
        writer,
        sheet_name="Extract Analysis",
        index=False
    )

    audit_summary.to_excel(
        writer,
        sheet_name="Audit Analysis",
        index=False
    )

    region_summary.to_excel(
        writer,
        sheet_name="Region Analysis",
        index=False
    )

    purpose_summary.to_excel(
        writer,
        sheet_name="Purpose Analysis",
        index=False
    )

    source_summary.to_excel(
        writer,
        sheet_name="Source Analysis",
        index=False
    )

print("\nAnalysis completed successfully.")
print("Output file:", output_file)

Dataset Shape: (600, 14)

Columns:
['Record_ID', 'Customer_ID', 'Event_Date', 'Data_Source', 'Purpose', 'Consent_Status', 'Consent_Date', 'Revocation_Date', 'Retention_Days', 'Deletion_Status', 'Extract_Status', 'Audit_Status', 'Region', 'Records_Processed']

Data Types:
Record_ID                    object
Customer_ID                  object
Event_Date           datetime64[ns]
Data_Source                  object
Purpose                      object
Consent_Status               object
Consent_Date         datetime64[ns]
Revocation_Date      datetime64[ns]
Retention_Days                int64
Deletion_Status              object
Extract_Status               object
Audit_Status                 object
Region                       object
Records_Processed             int64
dtype: object

Missing Values:
Record_ID              0
Customer_ID            0
Event_Date             0
Data_Source            0
Purpose                0
Consent_Status         0
Consent_Date         126
Revocation_Date   